In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

def descargar_datos(ticker, start_date, end_date):
    """Descarga los datos ajustados al cierre para un ticker específico y maneja errores."""
    try:
        ticker_data = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
        return ticker_data
    except Exception as e:
        print(f"Error descargando datos para {ticker}: {e}")
        return None

def calcular_modelo(ticker_data, market_data):
    """Calcula el modelo de regresión para un ticker y devuelve los resultados."""
    log_returns_ticker = np.log(ticker_data / ticker_data.shift(1)).dropna()
    log_returns_market = np.log(market_data / market_data.shift(1)).dropna()
    merged_data = pd.concat([log_returns_ticker, log_returns_market], axis=1).dropna()

    if merged_data.empty:
        return None

    X = sm.add_constant(merged_data.iloc[:, 1])
    y = merged_data.iloc[:, 0]

    modelo = sm.OLS(y, X).fit()
    resultados = {
        'Const': modelo.params['const'],
        'P-valor Const': modelo.pvalues['const'],
        'Beta': modelo.params[market_data.name],
        'P-valor Beta': modelo.pvalues[market_data.name],
        'R^2': modelo.rsquared
    }
    return resultados

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

market_index = "^GSPC"


for year in range(1998, 2023):
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"

    # Descarga de datos del mercado
    market_data = descargar_datos(market_index, start_date, end_date)

    if market_data is None or market_data.empty:
        print(f"No se obtuvieron datos del mercado para el año {year}")
        continue

    resultados_anuales = pd.DataFrame()

    for ticker in tickers:

        ticker_data = descargar_datos(ticker, start_date, end_date)

        if ticker_data is None or ticker_data.empty:
            print(f"No se obtuvieron datos para {ticker} en el año {year}")
            continue

        # Calcular el modelo de regresión
        resultados = calcular_modelo(ticker_data, market_data)

        if resultados:
            resultados_anuales.loc[ticker, 'Const'] = resultados['Const']
            resultados_anuales.loc[ticker, 'P-valor Const'] = resultados['P-valor Const']
            resultados_anuales.loc[ticker, 'Beta'] = resultados['Beta']
            resultados_anuales.loc[ticker, 'P-valor Beta'] = resultados['P-valor Beta']
            resultados_anuales.loc[ticker, 'R^2'] = resultados['R^2']
        else:
            print(f"No se pudo calcular el modelo para {ticker} en el año {year}")

    if not resultados_anuales.empty:
        # Guardar los resultados en un archivo CSV
        resultados_anuales.to_csv(f"resultados_{year}.csv")
        print(f"Resultados guardados para el año {year}")
    else:
        print(f"No se obtuvieron resultados válidos para el año {year}")